In [ ]:
from deltalake import DeltaTable

table_path = (
    "abfss://cb67eeef-0714-4dab-9b0f-6faea05bef24"
    "@onelake.dfs.fabric.microsoft.com/"
    "74682ba6-6374-4e0e-8489-c738f99dcf3b/Tables/dbo/station_raw"
)

storage_options = {
    "bearer_token": notebookutils.credentials.getToken("storage"),
    "use_fabric_endpoint": "true",
}

df = DeltaTable(table_path, storage_options=storage_options).to_pandas()

print(f"Relevés chargés : {len(df):,}")
display(df.head(10))

Relevés chargés : 60,706


In [30]:
import pandas as pd

clean = df.copy()

clean["station_id"] = clean["station_id"].astype("string").str.strip()
clean["collected_at"] = pd.to_datetime(
    clean["collected_at"], format="ISO8601", utc=True, errors="coerce"
)
clean["last_reported_at"] = pd.to_datetime(
    clean["last_reported"].where(clean["last_reported"] > 0),
    unit="s",
    utc=True,
    errors="coerce"
)

valid = (
    clean["station_id"].fillna("").ne("")
    & clean["station_id"].ne("TEST001")
    & clean["collected_at"].notna()
    & clean["num_bikes_available"].ge(0)
    & clean["num_docks_available"].ge(0)
)

clean = clean.loc[valid].drop_duplicates(
    subset=["station_id", "collected_at"]
)

print(f"Lignes conservées : {len(clean)} / {len(df)}")
display(clean[["station_id", "collected_at", "last_reported_at", "num_bikes_available"]])

Lignes conservées : 60706 / 60706


In [31]:
print(clean[["collected_at", "last_reported_at"]].dtypes)
print(clean.loc[:, ["collected_at", "last_reported_at"]].head(3).to_string(index=False))

collected_at        datetime64[ns, UTC]
last_reported_at    datetime64[ns, UTC]
dtype: object
                    collected_at          last_reported_at
2026-11-15 18:53:44.021287+00:00 2026-11-15 14:44:28+00:00
2026-11-15 18:53:44.021287+00:00 2026-11-15 14:36:14+00:00
2026-11-15 18:53:44.021287+00:00 2026-11-15 16:07:16+00:00


In [32]:
print(f"Relevés bruts : {len(df):,}")
print(f"Relevés conservés : {len(clean):,}")
print(f"Stations distinctes : {clean['station_id'].nunique():,}")

Relevés bruts : 60,706
Relevés conservés : 60,706
Stations distinctes : 466


In [33]:
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake

history_path = table_path.rsplit("/", 1)[0] + "/station_history"

history_data = clean.copy()
for column in ["collected_at", "last_reported_at"]:
    history_data[column] = history_data[column].astype("datetime64[us, UTC]")

batch = pa.Table.from_pandas(history_data, preserve_index=False)

storage_options = {
    "bearer_token": notebookutils.credentials.getToken("storage"),
    "use_fabric_endpoint": "true",
}

if DeltaTable.is_deltatable(history_path, storage_options=storage_options):
    history = DeltaTable(history_path, storage_options=storage_options)

    metrics = (
        history.merge(
            source=batch,
            source_alias="incoming",
            target_alias="existing",
            predicate=(
                "existing.station_id = incoming.station_id "
                "AND existing.collected_at = incoming.collected_at"
            ),
        )
        .when_not_matched_insert_all()
        .execute()
    )

    print(f"Nouveaux relevés ajoutés : {metrics['num_target_rows_inserted']:,}")
else:
    write_deltalake(
        history_path,
        batch,
        mode="error",
        storage_options=storage_options,
    )
    print(f"Historique créé avec {batch.num_rows:,} relevés.")

Nouveaux relevés ajoutés : 466


In [34]:
import pandas as pd

history = DeltaTable(history_path, storage_options=storage_options)
station_ids = history.to_pandas(columns=["station_id"])

display(pd.DataFrame({
    "nombre_de_releves": [len(station_ids)],
    "nombre_de_stations": [station_ids["station_id"].nunique()],
}))

In [35]:
latest_path = history_path.rsplit("/", 1)[0] + "/station_latest"

history_df = DeltaTable(
    history_path, storage_options=storage_options
).to_pandas()

latest = (
    history_df
    .sort_values("collected_at", ascending=False)
    .drop_duplicates(subset="station_id", keep="first")
)

for column in ["collected_at", "last_reported_at"]:
    latest[column] = latest[column].astype("datetime64[us, UTC]")

write_deltalake(
    latest_path,
    pa.Table.from_pandas(latest, preserve_index=False),
    mode="overwrite",
    storage_options=storage_options,
)

print(f"Stations dans station_latest : {len(latest):,}")

Stations dans station_latest : 466
